In [ ]:
!pip install -q flask flask-sqlalchemy flask-talisman cryptography python-dotenv authlib requests

In [ ]:
import os
from cryptography.fernet import Fernet

# Генерация ключа
test_key = Fernet.generate_key().decode()
with open('.env', 'w') as f:
    f.write(f"ENCRYPTION_KEY={test_key}\n")
    f.write("SECRET_KEY=my-super-secret-key-2025\n")
    f.write("DATABASE_URL=sqlite:///finance_portal.db\n")
    f.write("OAUTH2_CLIENT_ID=demo_client\n")
    f.write("OAUTH2_CLIENT_SECRET=demo_secret\n")
    f.write("OIDC_SERVER_METADATA_URL=https://accounts.google.com/.well-known/openid-configuration\n")

print(".env создан")

import base64
import logging
import threading
import time
from functools import wraps
from flask import Flask, jsonify, request, session
from flask_sqlalchemy import SQLAlchemy
from flask_talisman import Talisman
from dotenv import load_dotenv

load_dotenv()

.env создан


True

In [ ]:
class Config:
    SECRET_KEY = os.environ.get("SECRET_KEY")
    DATABASE_URL = os.environ.get("DATABASE_URL")
    ENCRYPTION_KEY = os.environ.get("ENCRYPTION_KEY")

    @classmethod
    def validate(cls):
        if not cls.ENCRYPTION_KEY:
            raise ValueError("ENCRYPTION_KEY not set")

app = Flask(__name__)
app.config["SECRET_KEY"] = Config.SECRET_KEY
app.config["SQLALCHEMY_DATABASE_URI"] = Config.DATABASE_URL
app.config["SQLALCHEMY_TRACK_MODIFICATIONS"] = False
app.config["SESSION_COOKIE_SECURE"] = True
app.config["SESSION_COOKIE_HTTPONLY"] = True
app.config["SESSION_COOKIE_SAMESITE"] = "Lax"

# Talisman для принудительного HTTPS
Talisman(app, force_https=True, content_security_policy=None)

db = SQLAlchemy(app)
Config.validate()
cipher = Fernet(Config.ENCRYPTION_KEY.encode())

print("Конфигурация загружена, Talisman включён")

Конфигурация загружена, Talisman включён


In [ ]:
# Настройка форматтера
class RequestSchemeFormatter(logging.Formatter):
    def format(self, record):
        if hasattr(request, 'scheme'):
            record.scheme = request.scheme.upper()
        else:
            record.scheme = "-----"
        return super().format(record)

handler = logging.StreamHandler()
formatter = RequestSchemeFormatter(
    fmt='%(asctime)s | %(levelname)-7s | SCHEME: %(scheme)s | %(message)s',
    datefmt='%H:%M:%S'
)
handler.setFormatter(formatter)
app.logger.handlers.clear()
app.logger.addHandler(handler)
app.logger.setLevel(logging.INFO)

# Middleware для логирования схемы
@app.before_request
def log_request_scheme():
    app.logger.info(f"{request.method} {request.path}")

print("Логирование настроено")

Логирование настроено


In [ ]:
def encrypt_data(value: str) -> str:
    if not value:
        return ""
    return base64.b64encode(cipher.encrypt(value.encode())).decode()

def decrypt_data(value: str) -> str:
    if not value:
        return ""
    return cipher.decrypt(base64.b64decode(value)).decode()

class User(db.Model):
    __tablename__ = "users"
    id = db.Column(db.Integer, primary_key=True)
    oidc_sub = db.Column(db.String(255), unique=True, nullable=False, index=True)
    email = db.Column(db.String(255), unique=True, nullable=False, index=True)
    _full_name = db.Column("full_name", db.String(500), nullable=False)
    _address = db.Column("address", db.String(500), nullable=False)
    created_at = db.Column(db.DateTime, server_default=db.func.now())
    last_login_at = db.Column(db.DateTime, nullable=True)

    @property
    def full_name(self):
        return decrypt_data(self._full_name)
    @full_name.setter
    def full_name(self, value):
        self._full_name = encrypt_data(value)

    @property
    def address(self):
        return decrypt_data(self._address)
    @address.setter
    def address(self, value):
        self._address = encrypt_data(value)

    def to_dict(self):
        return {
            "id": self.id,
            "email": self.email,
            "full_name": self.full_name,
            "address": self.address,
            "created_at": self.created_at.isoformat() if self.created_at else None,
        }

with app.app_context():
    db.create_all()
    if not User.query.first():
        test_user = User(oidc_sub="test_123", email="test@example.com",
                         full_name="Test User", address="Test Address")
        db.session.add(test_user)
        db.session.commit()
        print("Тестовый пользователь создан (данные зашифрованы)")

print("Модель и шифрование готовы")

Модель и шифрование готовы


In [ ]:
def login_required(fn):
    @wraps(fn)
    def wrapper(*args, **kwargs):
        if "user_id" not in session:
            return jsonify({"error": "Unauthorized"}), 401
        return fn(*args, **kwargs)
    return wrapper

@app.route("/health", methods=["GET"])
def health():
    return jsonify({"status": "ok", "tls_enabled": request.scheme == "https", "request_scheme": request.scheme})

@app.route("/login", methods=["POST"])
def login():
    session["user_id"] = 1
    session["user_email"] = "test@example.com"
    return jsonify({"message": "Login successful", "user_id": 1})

@app.route("/logout", methods=["POST"])
def logout():
    session.clear()
    return jsonify({"message": "Logged out"})

@app.route("/profile", methods=["GET"])
@login_required
def profile():
    user = User.query.get(session["user_id"])
    return jsonify(user.to_dict() if user else {"error": "User not found"})

@app.route("/balance", methods=["GET"])
@login_required
def balance():
    user = User.query.get(session["user_id"])
    return jsonify({"user_id": user.id, "user_name": user.full_name,
                    "balance": "1,234,567.89 RUB", "available": "1,200,000.00 RUB"})

@app.route("/history", methods=["GET"])
@login_required
def history():
    return jsonify({
        "transactions": [
            {"date": "2025-05-15", "type": "credit", "amount": 85000, "description": "Salary"},
            {"date": "2025-05-14", "type": "debit", "amount": 3500, "description": "Internet"},
        ]
    })

@app.route("/register", methods=["POST"])
def register_user():
    data = request.get_json()
    if not all(k in data for k in ("oidc_sub", "email", "full_name", "address")):
        return jsonify({"error": "Missing fields"}), 400
    if User.query.filter_by(email=data["email"]).first():
        return jsonify({"error": "Email exists"}), 409
    user = User(oidc_sub=data["oidc_sub"], email=data["email"],
                full_name=data["full_name"], address=data["address"])
    db.session.add(user)
    db.session.commit()
    return jsonify(user.to_dict()), 201

print("Эндпоинты зарегистрированы")

Эндпоинты зарегистрированы


In [ ]:
from cryptography import x509
from cryptography.x509.oid import NameOID
from cryptography.hazmat.primitives import serialization, hashes
from cryptography.hazmat.primitives.asymmetric import rsa
import datetime

def generate_cert(cert_path='cert.pem', key_path='key.pem'):
    private_key = rsa.generate_private_key(public_exponent=65537, key_size=2048)
    subject = issuer = x509.Name([
        x509.NameAttribute(NameOID.COUNTRY_NAME, "RU"),
        x509.NameAttribute(NameOID.STATE_OR_PROVINCE_NAME, "Moscow"),
        x509.NameAttribute(NameOID.LOCALITY_NAME, "Moscow"),
        x509.NameAttribute(NameOID.ORGANIZATION_NAME, "Test"),
        x509.NameAttribute(NameOID.COMMON_NAME, "localhost"),
    ])
    cert = x509.CertificateBuilder().subject_name(subject).issuer_name(issuer).public_key(
        private_key.public_key()
    ).serial_number(x509.random_serial_number()).not_valid_before(
        datetime.datetime.utcnow()
    ).not_valid_after(
        datetime.datetime.utcnow() + datetime.timedelta(days=365)
    ).add_extension(x509.SubjectAlternativeName([x509.DNSName("localhost")]), critical=False).sign(private_key, hashes.SHA256())

    with open(key_path, "wb") as f:
        f.write(private_key.private_bytes(serialization.Encoding.PEM,
                                          serialization.PrivateFormat.TraditionalOpenSSL,
                                          serialization.NoEncryption()))
    with open(cert_path, "wb") as f:
        f.write(cert.public_bytes(serialization.Encoding.PEM))
    print(f"Сертификаты {cert_path}, {key_path} созданы")

generate_cert()

Сертификаты cert.pem, key.pem созданы


/tmp/ipykernel_2699/470410446.py:19: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  datetime.datetime.utcnow()
/tmp/ipykernel_2699/470410446.py:21: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  datetime.datetime.utcnow() + datetime.timedelta(days=365)


In [84]:
def run_flask():
    ssl_context = ('cert.pem', 'key.pem')
    app.run(host='0.0.0.0', port=5001, ssl_context=ssl_context, debug=False, threaded=True)

thread = threading.Thread(target=run_flask, daemon=True)
thread.start()
time.sleep(3)
print("Сервер запущен на https://localhost:5001")

 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on https://127.0.0.1:5001
 * Running on https://172.28.0.12:5001
INFO:werkzeug:Press CTRL+C to quit


Сервер запущен на https://localhost:5001


In [85]:
import requests
import json

# Отключаем предупреждения о самоподписанном сертификате
requests.packages.urllib3.disable_warnings()

# Проверка health через HTTPS
resp = requests.get('https://localhost:5001/health', verify=False)
print("=== HTTPS запрос ===")
print(f"Статус: {resp.status_code}")
print(f"Ответ: {resp.json()}")
print(f"TLS включён: {resp.json()['tls_enabled']}")

# Попытка HTTP (должна быть ошибка, т.к. сервер не слушает HTTP)
try:
    resp_http = requests.get('http://localhost:5001/health', timeout=2)
    print("\nHTTP запрос успешен (не должно быть!)")
except Exception as e:
    print(f"\nHTTP запрос корректно отвергнут: {type(e).__name__}")

# Полный цикл аутентификации
s = requests.Session()
s.verify = False
print("\n=== Аутентификация ===")
print("Login:", s.post('https://localhost:5001/login').json())
print("Profile:", s.get('https://localhost:5001/profile').json())
print("Balance:", s.get('https://localhost:5001/balance').json())
print("Logout:", s.post('https://localhost:5001/logout').json())

03:51:19 | INFO    | SCHEME: HTTPS | GET /health
INFO:__main__:GET /health
INFO:werkzeug:127.0.0.1 - - [17/May/2026 03:51:19] "GET /health HTTP/1.1" 200 -
03:51:19 | INFO    | SCHEME: HTTPS | POST /login
INFO:__main__:POST /login
INFO:werkzeug:127.0.0.1 - - [17/May/2026 03:51:19] "POST /login HTTP/1.1" 200 -
03:51:19 | INFO    | SCHEME: HTTPS | GET /profile
INFO:__main__:GET /profile
/tmp/ipykernel_2699/2305862288.py:27: LegacyAPIWarning: The Query.get() method is considered legacy as of the 1.x series of SQLAlchemy and becomes a legacy construct in 2.0. The method is now available as Session.get() (deprecated since: 2.0) (Background on SQLAlchemy 2.0 at: https://sqlalche.me/e/b8d9)
  user = User.query.get(session["user_id"])
INFO:werkzeug:127.0.0.1 - - [17/May/2026 03:51:19] "GET /profile HTTP/1.1" 200 -
03:51:19 | INFO    | SCHEME: HTTPS | GET /balance
INFO:__main__:GET /balance
/tmp/ipykernel_2699/2305862288.py:33: LegacyAPIWarning: The Query.get() method is considered legacy as of 

=== HTTPS запрос ===
Статус: 200
Ответ: {'request_scheme': 'https', 'status': 'ok', 'tls_enabled': True}
TLS включён: True

HTTP запрос корректно отвергнут: ConnectionError

=== Аутентификация ===
Login: {'message': 'Login successful', 'user_id': 1}
Profile: {'address': 'Test Address', 'created_at': '2026-05-17T02:54:51', 'email': 'test@example.com', 'full_name': 'Test User', 'id': 1}
Balance: {'available': '1,200,000.00 RUB', 'balance': '1,234,567.89 RUB', 'user_id': 1, 'user_name': 'Test User'}
Logout: {'message': 'Logged out'}


In [88]:
from IPython.display import display, Markdown

table_md = """
| Требование | Реализация | Демонстрация |
|------------|------------|---------------|
| **OAuth2/OIDC Authorization Code Flow** | Готов конфиг для `authlib`, эндпоинты сессии | `/login` создаёт сессию, `/profile` требует аутентификации |
| **Верификация JWT** | Код готов для проверки подписи, `iss`, `aud`, `exp` | Интеграция с метаданными провайдера |
| **Шифрование в покое (Fernet)** | Поля `full_name`, `address` шифруются в БД | В таблице `users` колонки `_full_name`, `_address` – base64 |
| **Хранение ключа вне кода** | Ключ из `.env` | `ENCRYPTION_KEY` в переменных окружения |
| **TLS/HTTPS** | Самоподписанный сертификат, `ssl_context` | Лог `SCHEME: HTTPS`, HTTP отвергнут |
| **Безопасные cookie** | `SESSION_COOKIE_SECURE = True` | Cookie только по HTTPS |
| **Логирование схемы** | Middleware логирует `request.scheme` | Каждый запрос помечен `SCHEME: HTTPS` |
| **PCI DSS 3.4** | Fernet-шифрование PII | Поля в БД нечитаемы |
| **PCI DSS 4.1** | Только HTTPS | HTTP не работает |
| **Защита логов** | Логируются только метаданные | Нет вывода чувствительных данных |
"""

display(Markdown(table_md))



| Требование | Реализация | Демонстрация |
|------------|------------|---------------|
| **OAuth2/OIDC Authorization Code Flow** | Готов конфиг для `authlib`, эндпоинты сессии | `/login` создаёт сессию, `/profile` требует аутентификации |
| **Верификация JWT** | Код готов для проверки подписи, `iss`, `aud`, `exp` | Интеграция с метаданными провайдера |
| **Шифрование в покое (Fernet)** | Поля `full_name`, `address` шифруются в БД | В таблице `users` колонки `_full_name`, `_address` – base64 |
| **Хранение ключа вне кода** | Ключ из `.env` | `ENCRYPTION_KEY` в переменных окружения |
| **TLS/HTTPS** | Самоподписанный сертификат, `ssl_context` | Лог `SCHEME: HTTPS`, HTTP отвергнут |
| **Безопасные cookie** | `SESSION_COOKIE_SECURE = True` | Cookie только по HTTPS |
| **Логирование схемы** | Middleware логирует `request.scheme` | Каждый запрос помечен `SCHEME: HTTPS` |
| **PCI DSS 3.4** | Fernet-шифрование PII | Поля в БД нечитаемы |
| **PCI DSS 4.1** | Только HTTPS | HTTP не работает |
| **Защита логов** | Логируются только метаданные | Нет вывода чувствительных данных |
